# 00 — Definición de la Tabla Base

**Objetivo**: Analizar las definiciones de las 4 fuentes disponibles en `dataset.json` para:
1. Entender qué almacena cada fuente y a qué nivel de granularidad opera
2. Identificar la **fuente madre** (tabla base) del modelo predictivo
3. Mapear las llaves de integración entre fuentes
4. Clasificar columnas por rol predictivo (feature vs target vs leakage)

**Contexto**: Modelo predictivo sobre contratos de Obra Pública en SECOP II.  
**Metodología**: CRISP-DM — Fase 2: Comprensión de los datos.

In [1]:
import json
import pandas as pd
from pathlib import Path

# Cargar definiciones
DATASET_PATH = Path("../dataset.json")
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

fuentes = dataset["fuentes"]
print(f"Fuentes disponibles: {list(fuentes.keys())}")
print(f"App token: {dataset['app_token'][:8]}...")

Fuentes disponibles: ['contratosElectronicos', 'proveedoresRegistrados', 'procesosDeContratacion', 'adiciones']
App token: nrnMDVHr...


## 1. Resumen de cada fuente

Extraemos la definición oficial y el conteo de columnas por fuente.

In [2]:
resumen = []
for nombre, info in fuentes.items():
    cols = info["columnas"]
    tipos = pd.Series([c["tipo"] for c in cols]).value_counts().to_dict()
    resumen.append({
        "Fuente": nombre,
        "API (dataset ID)": info["apiSODA3"].split("/views/")[1].split("/")[0],
        "Columnas": len(cols),
        "Texto": tipos.get("Texto", 0),
        "Número": tipos.get("Número", 0),
        "Timestamp": tipos.get("Marca de tiempo variable", 0) + tipos.get("Marca de tiempo", 0),
        "URL": tipos.get("URL", 0),
    })

df_resumen = pd.DataFrame(resumen)
df_resumen

,Fuente,API (dataset ID),Columnas,Texto,Número,Timestamp,URL
0,contratosElectronicos,jbjy-vk9h,87,60,19,7,1
1,proveedoresRegistrados,qmzu-gj57,25,24,0,1,0
2,procesosDeContratacion,p6dx-8zbt,59,33,14,11,1
3,adiciones,cb9c-h8sn,5,4,0,1,0


In [3]:
# Definiciones oficiales de cada fuente
for nombre, info in fuentes.items():
    print(f"\n{'='*80}")
    print(f"FUENTE: {nombre}")
    print(f"{'='*80}")
    print(info["definicion"])


FUENTE: contratosElectronicos
El conjunto de datos SECOP II - Contratos Electrónicos contiene información detallada sobre los contratos estatales firmados electrónicamente en Colombia. Incluye variables relacionadas con:
 - Identificación de la entidad contratante (nombre, NIT, ubicación, sector, orden, rama)
 - Información del contrato (ID, referencia, estado, tipo, modalidad, fechas clave, objeto, duración)
 - Datos del proveedor adjudicado (nombre, documento, tipo, grupo, Pyme)
 - Información financiera (valores contratados, pagos, amortizaciones, saldos, origen y destino de recursos)
 - Información presupuestal (códigos BPIN, CDP, vigencia)
 - Datos del representante legal, supervisores, ordenadores de gasto y pago
 - Indicadores de cumplimiento ambiental, postconsumo, postconflicto y prórrogas
 Este conjunto permite realizar análisis sobre la ejecución contractual, seguimiento presupuestal, evaluación de proveedores, transparencia en la gestión pública y apoyo a políticas de cont

## 2. Análisis de columnas por fuente

Construimos un DataFrame con todas las columnas de todas las fuentes para poder cruzarlas.

In [4]:
# DataFrame consolidado de todas las columnas
todas_columnas = []
for nombre_fuente, info in fuentes.items():
    for col in info["columnas"]:
        todas_columnas.append({
            "fuente": nombre_fuente,
            "columna": col["nombreColumna"],
            "campo_api": col["nombreCampoApi"],
            "tipo": col["tipo"],
            "descripcion": col["descripcion"],
        })

df_cols = pd.DataFrame(todas_columnas)
print(f"Total columnas registradas: {len(df_cols)}")
df_cols.groupby("fuente")["columna"].count().sort_values(ascending=False)

Total columnas registradas: 176


fuente
contratosElectronicos     87
procesosDeContratacion    59
proveedoresRegistrados    25
adiciones                  5
Name: columna, dtype: int64

## 3. Diccionario de columnas por fuente

Mostramos el diccionario completo de cada fuente para entender qué información captura.

In [5]:
# --- contratosElectronicos (jbjy-vk9h) ---
print("CONTRATOS ELECTRÓNICOS — Columnas")
print("="*80)
df_ce = df_cols[df_cols["fuente"] == "contratosElectronicos"][["columna", "campo_api", "tipo", "descripcion"]]
df_ce = df_ce.reset_index(drop=True)
df_ce.index += 1
with pd.option_context("display.max_colwidth", 100, "display.max_rows", 100):
    display(df_ce)

CONTRATOS ELECTRÓNICOS — Columnas


,columna,campo_api,tipo,descripcion
1,Nombre Entidad,nombre_entidad,Texto,Nombre de la entidad del estado que publica el contrato
2,Nit Entidad,nit_entidad,Número,NIT de la entidad del estado que publica el contrato
3,Departamento,departamento,Texto,Departamento en el cual se registró la entidad del estado que publica el contrato
4,Ciudad,ciudad,Texto,Ciudad en el cual se registró la entidad del estado que publica el contrato
5,Localización,localizaci_n,Texto,Ubicación completa de la entidad del estado que publica el contrato
6,Orden,orden,Texto,Orden entidad del estado que publica el contrato
7,Sector,sector,Texto,Sector entidad del estado que publica el contrato
8,Rama,rama,Texto,Rama del estado de la entidad que publica el contrato
9,Entidad Centralizada,entidad_centralizada,Texto,Define si la entidad es descentralizada o centralizada
10,Proceso de Compra,proceso_de_compra,Texto,Identificador del proceso de compra publicado


In [6]:
# --- procesosDeContratacion (p6dx-8zbt) ---
print("PROCESOS DE CONTRATACIÓN — Columnas")
print("="*80)
df_pc = df_cols[df_cols["fuente"] == "procesosDeContratacion"][["columna", "campo_api", "tipo", "descripcion"]]
df_pc = df_pc.reset_index(drop=True)
df_pc.index += 1
with pd.option_context("display.max_colwidth", 100, "display.max_rows", 100):
    display(df_pc)

PROCESOS DE CONTRATACIÓN — Columnas


,columna,campo_api,tipo,descripcion
1,Entidad,entidad,Texto,Nombre de la Entidad que publica el proceso de compra pública
2,Nit Entidad,nit_entidad,Texto,NIT de la Entidad que publicó el proceso
3,Departamento Entidad,departamento_entidad,Texto,Departamento en el cual está registrada la entidad
4,Ciudad Entidad,ciudad_entidad,Texto,Ciudad en la cual está registrada la entidad
5,Orden Entidad,ordenentidad,Texto,"Orden de la Entidad (Nacional, Regional)"
6,Entidad Centralizada,codigo_pci,Texto,Identifica si la entidad es o no centralizada
7,ID del Proceso,id_del_proceso,Texto,"Identificador Único del Proceso, valor generado por la plataforma"
8,Referencia del Proceso,referencia_del_proceso,Texto,"Identificador del Proceso, valor generado por la Entidad"
9,PCI,ppi,Texto,Código de Unidad - Sub-Unidad Contratación
10,ID del Portafolio,id_del_portafolio,Texto,Identificador del Portafolio al cual corresponde el proceso de compra


In [7]:
# --- adiciones (cb9c-h8sn) ---
print("ADICIONES — Columnas")
print("="*80)
df_ad = df_cols[df_cols["fuente"] == "adiciones"][["columna", "campo_api", "tipo", "descripcion"]]
df_ad = df_ad.reset_index(drop=True)
df_ad.index += 1
with pd.option_context("display.max_colwidth", 100, "display.max_rows", 100):
    display(df_ad)

ADICIONES — Columnas


,columna,campo_api,tipo,descripcion
1,Identificador,identificador,Texto,Identificador único del evento de modificación
2,ID Contrato,Id contrato,Texto,Identificador del contrato al que corresponde la modificación
3,Tipo,tipo,Texto,"Tipo de modificación, de acuerdo con el impacto que tiene sobre el contrato"
4,Descripción,descripción,Texto,Descripción detallada de la justificación de la adición o modificación
5,Fecha Registro,Fecha registro,Marca de tiempo,Fecha en que se hace la modificación


In [8]:
# --- proveedoresRegistrados (qmzu-gj57) ---
print("PROVEEDORES REGISTRADOS — Columnas")
print("="*80)
df_pv = df_cols[df_cols["fuente"] == "proveedoresRegistrados"][["columna", "campo_api", "tipo", "descripcion"]]
df_pv = df_pv.reset_index(drop=True)
df_pv.index += 1
with pd.option_context("display.max_colwidth", 100, "display.max_rows", 100):
    display(df_pv)

PROVEEDORES REGISTRADOS — Columnas


,columna,campo_api,tipo,descripcion
1,Código,codigo,Texto,Es el código del proveedor en la plataforma SECOPII
2,Nombre,nombre,Texto,"Nombre del Proveedor, como se registró en SECOP II"
3,NIT,nit,Texto,Número de Identificación con el que figura el proveedor en SECOP II
4,Es Entidad,es_entidad,Texto,Indica si el proveedor también es una entidad compradora en SECOPII
5,Es grupo,es_grupo,Texto,Indica si la entidad pertenece a un grupo
6,Esta Activa,esta_activa,Texto,Indica si el proveedor esta activo en el SECOPII
7,Fecha Creación,fecha_creacion,Marca de tiempo variable,Fecha en la que se hizo el primer registro del proveedor
8,Código Categoría Principal,codigo_categoria_principal,Texto,Código UNSPSC principal del proveedor
9,Descripción Categoría Principal,descripcion_categoria_principal,Texto,Descripción UNSPSC principal del proveedor
10,Teléfono,teléfono,Texto,Indica el número de teléfono del proveedor


## 4. Identificación de llaves de integración

Buscamos campos compartidos entre fuentes que podrían servir como llaves de unión.

In [9]:
# Campos API compartidos entre fuentes
from itertools import combinations

campos_por_fuente = {}
for nombre, info in fuentes.items():
    campos_por_fuente[nombre] = set(c["nombreCampoApi"] for c in info["columnas"])

print("CAMPOS COMPARTIDOS ENTRE FUENTES")
print("="*80)
for f1, f2 in combinations(campos_por_fuente.keys(), 2):
    comunes = campos_por_fuente[f1] & campos_por_fuente[f2]
    if comunes:
        print(f"\n{f1} ∩ {f2} ({len(comunes)} campos):")
        for c in sorted(comunes):
            print(f"  - {c}")
    else:
        print(f"\n{f1} ∩ {f2}: SIN campos compartidos")

CAMPOS COMPARTIDOS ENTRE FUENTES

contratosElectronicos ∩ proveedoresRegistrados (3 campos):
  - departamento
  - es_grupo
  - nombre_representante_legal

contratosElectronicos ∩ procesosDeContratacion (5 campos):
  - codigo_entidad
  - modalidad_de_contratacion
  - nit_entidad
  - tipo_de_contrato
  - urlproceso

contratosElectronicos ∩ adiciones: SIN campos compartidos

proveedoresRegistrados ∩ procesosDeContratacion: SIN campos compartidos

proveedoresRegistrados ∩ adiciones: SIN campos compartidos

procesosDeContratacion ∩ adiciones: SIN campos compartidos


In [10]:
# Mapa de llaves de integración propuesto
llaves = [
    {
        "Relación": "contratosElectronicos → procesosDeContratacion",
        "Campo CE": "proceso_de_compra",
        "Campo PC": "id_del_proceso",
        "Tipo": "FK → PK",
        "Descripción": "Cada contrato electrónico nace de un proceso de compra",
        "Confiabilidad": "ALTA",
    },
    {
        "Relación": "contratosElectronicos → proveedoresRegistrados",
        "Campo CE": "codigo_proveedor",
        "Campo PC": "codigo",
        "Tipo": "FK → PK",
        "Descripción": "El proveedor adjudicado está en el registro de proveedores",
        "Confiabilidad": "ALTA",
    },
    {
        "Relación": "adiciones → contratosElectronicos",
        "Campo CE": "Id contrato",
        "Campo PC": "id_contrato",
        "Tipo": "FK → PK",
        "Descripción": "Cada adición/modificación referencia un contrato",
        "Confiabilidad": "ALTA",
    },
    {
        "Relación": "procesosDeContratacion → proveedoresRegistrados",
        "Campo CE": "codigoproveedor",
        "Campo PC": "codigo",
        "Tipo": "FK → PK",
        "Descripción": "El proveedor adjudicado en el proceso",
        "Confiabilidad": "ALTA",
    },
    {
        "Relación": "contratosElectronicos ↔ procesosDeContratacion (entidad)",
        "Campo CE": "codigo_entidad / nit_entidad",
        "Campo PC": "codigo_entidad / nit_entidad",
        "Tipo": "Compartido",
        "Descripción": "Identificación de la entidad contratante",
        "Confiabilidad": "ALTA",
    },
]

df_llaves = pd.DataFrame(llaves)
with pd.option_context("display.max_colwidth", 60):
    display(df_llaves)

,Relación,Campo CE,Campo PC,Tipo,Descripción,Confiabilidad
0,contratosElectronicos → procesosDeContratacion,proceso_de_compra,id_del_proceso,FK → PK,Cada contrato electrónico nace de un proceso de compra,ALTA
1,contratosElectronicos → proveedoresRegistrados,codigo_proveedor,codigo,FK → PK,El proveedor adjudicado está en el registro de proveedores,ALTA
2,adiciones → contratosElectronicos,Id contrato,id_contrato,FK → PK,Cada adición/modificación referencia un contrato,ALTA
3,procesosDeContratacion → proveedoresRegistrados,codigoproveedor,codigo,FK → PK,El proveedor adjudicado en el proceso,ALTA
4,contratosElectronicos ↔ procesosDeContratacion (entidad),codigo_entidad / nit_entidad,codigo_entidad / nit_entidad,Compartido,Identificación de la entidad contratante,ALTA


## 5. Clasificación funcional de columnas

Para el modelo predictivo de Obra Pública, clasificamos cada columna según su rol:
- **ID/Llave**: Identificadores y llaves de integración
- **Feature - Entidad**: Características de la entidad contratante
- **Feature - Proceso**: Info del proceso de selección (pre-contrato)
- **Feature - Proveedor**: Características del proveedor
- **Feature - Contractual**: Info del contrato firmado (temporal, financiera)
- **Target potencial**: Variables que miden el resultado/desempeño
- **Leakage**: Variables que solo existen después del evento a predecir
- **Descartable**: Info no relevante para ML (URLs, nombres de personas, cuentas bancarias)

In [11]:
# === CLASIFICACIÓN DE contratosElectronicos ===
clasificacion_ce = {
    # --- ID / Llaves ---
    "id_contrato":             "ID/Llave",
    "proceso_de_compra":       "ID/Llave",
    "referencia_del_contrato": "ID/Llave",
    "codigo_entidad":          "ID/Llave",
    "codigo_proveedor":        "ID/Llave",
    "nit_entidad":             "ID/Llave",
    
    # --- Feature - Entidad ---
    "nombre_entidad":          "Feature - Entidad",
    "departamento":            "Feature - Entidad",
    "ciudad":                  "Feature - Entidad",
    "localizaci_n":            "Feature - Entidad",
    "orden":                   "Feature - Entidad",
    "sector":                  "Feature - Entidad",
    "rama":                    "Feature - Entidad",
    "entidad_centralizada":    "Feature - Entidad",
    
    # --- Feature - Contractual ---
    "tipo_de_contrato":                "Feature - Contractual",
    "modalidad_de_contratacion":       "Feature - Contractual",
    "justificacion_modalidad_de":      "Feature - Contractual",
    "codigo_de_categoria_principal":   "Feature - Contractual",
    "descripcion_del_proceso":         "Feature - Contractual",
    "objeto_del_contrato":             "Feature - Contractual",
    "condiciones_de_entrega":          "Feature - Contractual",
    "duraci_n_del_contrato":           "Feature - Contractual",
    "fecha_de_firma":                  "Feature - Contractual",
    "fecha_de_inicio_del_contrato":    "Feature - Contractual",
    "fecha_de_fin_del_contrato":       "Feature - Contractual",
    "valor_del_contrato":              "Feature - Contractual",
    "origen_de_los_recursos":          "Feature - Contractual",
    "destino_gasto":                   "Feature - Contractual",
    "habilita_pago_adelantado":        "Feature - Contractual",
    "valor_de_pago_adelantado":        "Feature - Contractual",
    "el_contrato_puede_ser_prorrogado": "Feature - Contractual",
    "espostconflicto":                 "Feature - Contractual",
    "puntos_del_acuerdo":              "Feature - Contractual",
    "pilares_del_acuerdo":             "Feature - Contractual",
    "estado_bpin":                     "Feature - Contractual",
    "c_digo_bpin":                     "Feature - Contractual",
    "anno_bpin":                       "Feature - Contractual",
    "documentos_tipo":                 "Feature - Contractual",
    "descripcion_documentos_tipo":     "Feature - Contractual",
    
    # --- Feature - Presupuestal (origen de recursos) ---
    "presupuesto_general_de_la_nacion_pgn":  "Feature - Presupuestal",
    "sistema_general_de_participaciones":     "Feature - Presupuestal",
    "sistema_general_de_regal_as":            "Feature - Presupuestal",
    "recursos_propios_alcald_as_gobernaciones_y_resguardos_ind_genas_": "Feature - Presupuestal",
    "recursos_de_credito":                    "Feature - Presupuestal",
    "recursos_propios":                       "Feature - Presupuestal",
    
    # --- Feature - Proveedor ---
    "proveedor_adjudicado":    "Feature - Proveedor",
    "documento_proveedor":     "Feature - Proveedor",
    "tipodocproveedor":        "Feature - Proveedor",
    "es_grupo":                "Feature - Proveedor",
    "es_pyme":                 "Feature - Proveedor",
    
    # --- TARGET POTENCIAL ---
    "dias_adicionados":                "TARGET potencial",  # Días extra añadidos al contrato
    "liquidaci_n":                     "TARGET potencial",  # Si fue liquidado
    "estado_contrato":                 "TARGET potencial",  # Estado final del contrato
    
    # --- LEAKAGE (info post-evento) ---
    "valor_facturado":                 "LEAKAGE",
    "valor_pendiente_de_pago":         "LEAKAGE",
    "valor_pagado":                    "LEAKAGE",
    "valor_amortizado":                "LEAKAGE",
    "valor_pendiente_de":              "LEAKAGE",
    "valor_pendiente_de_ejecucion":    "LEAKAGE",
    "saldo_cdp":                       "LEAKAGE",
    "saldo_vigencia":                  "LEAKAGE",
    "fecha_inicio_liquidacion":        "LEAKAGE",
    "fecha_fin_liquidacion":           "LEAKAGE",
    "fecha_de_notificaci_n_de_prorrogaci_n": "LEAKAGE",
    "ultima_actualizacion":            "LEAKAGE",
    "obligaci_n_ambiental":            "LEAKAGE",
    "obligaciones_postconsumo":        "LEAKAGE",
    "reversion":                       "LEAKAGE",
    
    # --- DESCARTABLE (no relevante para ML) ---
    "urlproceso":                                    "Descartable",
    "nombre_representante_legal":                    "Descartable",
    "nacionalidad_representante_legal":              "Descartable",
    "domicilio_representante_legal":                 "Descartable",
    "tipo_de_identificaci_n_representante_legal":    "Descartable",
    "identificaci_n_representante_legal":            "Descartable",
    "g_nero_representante_legal":                    "Descartable",
    "nombre_ordenador_del_gasto":                    "Descartable",
    "tipo_de_documento_ordenador_del_gasto":         "Descartable",
    "n_mero_de_documento_ordenador_del_gasto":       "Descartable",
    "nombre_supervisor":                             "Descartable",
    "tipo_de_documento_supervisor":                  "Descartable",
    "n_mero_de_documento_supervisor":                "Descartable",
    "nombre_ordenador_de_pago":                      "Descartable",
    "tipo_de_documento_ordenador_de_pago":           "Descartable",
    "n_mero_de_documento_ordenador_de_pago":         "Descartable",
    "nombre_del_banco":                              "Descartable",
    "tipo_de_cuenta":                                "Descartable",
    "n_mero_de_cuenta":                              "Descartable",
}

# Construir tabla de clasificación
filas = []
for col_info in fuentes["contratosElectronicos"]["columnas"]:
    campo = col_info["nombreCampoApi"]
    filas.append({
        "Columna": col_info["nombreColumna"],
        "Campo API": campo,
        "Tipo": col_info["tipo"],
        "Rol": clasificacion_ce.get(campo, "SIN CLASIFICAR"),
        "Descripción": col_info["descripcion"][:80],
    })

df_clasif_ce = pd.DataFrame(filas)
print("\nRESUMEN DE CLASIFICACIÓN — contratosElectronicos")
print(df_clasif_ce["Rol"].value_counts())
print(f"\nColumnas sin clasificar: {(df_clasif_ce['Rol'] == 'SIN CLASIFICAR').sum()}")


RESUMEN DE CLASIFICACIÓN — contratosElectronicos
Rol
Feature - Contractual     25
Descartable               19
LEAKAGE                   15
Feature - Entidad          8
ID/Llave                   6
Feature - Presupuestal     6
Feature - Proveedor        5
TARGET potencial           3
Name: count, dtype: int64

Columnas sin clasificar: 0


In [12]:
# Mostrar clasificación completa
with pd.option_context("display.max_colwidth", 80, "display.max_rows", 70):
    display(df_clasif_ce.sort_values("Rol"))

,Columna,Campo API,Tipo,Rol,Descripción
72,Tipo de cuenta,tipo_de_cuenta,Texto,Descartable,Tipo de cuenta en la que se se consignaran los pagos del contrato
57,Género Representante Legal,g_nero_representante_legal,Texto,Descartable,Género sexual del representante legal de la entidad
54,Domicilio Representante Legal,domicilio_representante_legal,Texto,Descartable,Dirección de domicilio del representante legal de la entidad
53,Nacionalidad Representante Legal,nacionalidad_representante_legal,Texto,Descartable,Nacionalidad del representante legal de la entidad
71,Nombre del banco,nombre_del_banco,Texto,Descartable,Indica el nombre del banco en donde se consignarán los pagos del contrato
...,...,...,...,...,...
46,Saldo Vigencia,saldo_vigencia,Número,LEAKAGE,Saldo actual para la vigencia del CDP asignado al proceso y al contrato
75,Fecha de notificación de prorrogación,fecha_de_notificaci_n_de_prorrogaci_n,Marca de tiempo variable,LEAKAGE,Infica la fecha de notificación de la prorrogación del contrato
28,Liquidación,liquidaci_n,Texto,TARGET potencial,Determina si el contrato ha sido liquidado
48,Días adicionados,dias_adicionados,Número,TARGET potencial,Días adicionados al contrato electrónico


## 6. Determinación de la Fuente Madre

### Criterios de selección

La **fuente madre** debe ser la tabla cuya granularidad coincida con la **unidad de análisis** del modelo predictivo.

| Criterio | contratosElectronicos | procesosDeContratacion | adiciones | proveedores |
|---|---|---|---|---|
| Granularidad | 1 fila = 1 contrato firmado | 1 fila = 1 proceso de compra | 1 fila = 1 evento de modificación | 1 fila = 1 proveedor |
| Contiene targets | **SÍ** (dias_adicionados, estado_contrato, liquidación) | NO | Indirectamente (tipo de modificación) | NO |
| Contiene features pre-contrato | Parcialmente | **SÍ** (completo) | NO | **SÍ** (perfil proveedor) |
| Contiene datos de ejecución | **SÍ** (pagos, facturación, saldos) | NO | NO | NO |
| Relación con el contrato | **ES** el contrato | Proceso que genera el contrato | Modificaciones al contrato | Proveedor del contrato |

In [13]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    DECISIÓN: FUENTE MADRE                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Fuente madre: contratosElectronicos (jbjy-vk9h)                   ║
║                                                                    ║
║  Unidad de análisis: EL CONTRATO FIRMADO                           ║
║                                                                    ║
║  Justificación:                                                    ║
║  1. El contrato es la unidad sobre la cual ocurren los eventos     ║
║     que queremos predecir (atrasos, extensiones, sobrecostos)      ║
║  2. Contiene las variables TARGET directas:                        ║
║     - dias_adicionados → extensiones de plazo                      ║
║     - estado_contrato  → estado final                              ║
║     - liquidación      → cierre del contrato                       ║
║  3. Contiene datos de ejecución financiera que permiten            ║
║     calcular sobrecostos (valor_contrato vs valor_pagado)          ║
║  4. Tiene FK a procesos (proceso_de_compra) y proveedores          ║
║     (codigo_proveedor) para enriquecer con features                ║
║                                                                    ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Esquema relacional:                                               ║
║                                                                    ║
║  procesosDeContratacion ──(1:N)──► contratosElectronicos           ║
║  (proceso previo)         id_del_proceso = proceso_de_compra       ║
║                                       │                            ║
║  proveedoresRegistrados ──(1:N)──►    │  (tabla base / madre)      ║
║  (perfil proveedor)       codigo = codigo_proveedor                ║
║                                       │                            ║
║  adiciones ──────────────(N:1)──►     │                            ║
║  (modificaciones)         Id contrato = id_contrato                ║
║                                                                    ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Variables TARGET propuestas:                                      ║
║                                                                    ║
║  1. tiene_extension (binaria):                                     ║
║     dias_adicionados > 0                                           ║
║                                                                    ║
║  2. tiene_adicion_valor (binaria):                                 ║
║     COUNT(adiciones WHERE tipo='ADICION EN EL VALOR') > 0          ║
║                                                                    ║
║  3. sobrecosto_ratio (continua):                                   ║
║     (valor_pagado - valor_del_contrato) / valor_del_contrato       ║
║                                                                    ║
║  4. fue_liquidado (binaria):                                       ║
║     liquidación == 'Si'                                            ║
║                                                                    ║
║  5. riesgo_contractual (multiclase):                               ║
║     Combinación de extensiones + adiciones + sobrecostos           ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════╗
║                    DECISIÓN: FUENTE MADRE                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  Fuente madre: contratosElectronicos (jbjy-vk9h)                   ║
║                                                                    ║
║  Unidad de análisis: EL CONTRATO FIRMADO                           ║
║                                                                    ║
║  Justificación:                                                    ║
║  1. El contrato es la unidad sobre la cual ocurren los eventos     ║
║     que queremos predecir (atrasos, extensiones, sobrecostos)      ║
║  2. Contiene las variables TARGET directas:                        ║
║     - dias_adicionados → extensiones de plazo                      ║
║     - estado_contrato  → estado final                              ║
║

## 7. Columnas compartidas: solapamiento entre fuentes

Verificamos qué columnas aparecen en múltiples fuentes (por nombre de campo API).

In [14]:
# Columnas que aparecen en más de una fuente
campo_fuentes = df_cols.groupby("campo_api")["fuente"].apply(list).reset_index()
campo_fuentes["n_fuentes"] = campo_fuentes["fuente"].apply(len)
compartidos = campo_fuentes[campo_fuentes["n_fuentes"] > 1].sort_values("n_fuentes", ascending=False)

print(f"Campos presentes en más de 1 fuente: {len(compartidos)}")
for _, row in compartidos.iterrows():
    fuentes_str = ", ".join(row["fuente"])
    print(f"  {row['campo_api']:40s} → [{fuentes_str}]")

Campos presentes en más de 1 fuente: 8
  codigo_entidad                           → [contratosElectronicos, procesosDeContratacion]
  departamento                             → [contratosElectronicos, proveedoresRegistrados]
  es_grupo                                 → [contratosElectronicos, proveedoresRegistrados]
  modalidad_de_contratacion                → [contratosElectronicos, procesosDeContratacion]
  nit_entidad                              → [contratosElectronicos, procesosDeContratacion]
  nombre_representante_legal               → [contratosElectronicos, proveedoresRegistrados]
  tipo_de_contrato                         → [contratosElectronicos, procesosDeContratacion]
  urlproceso                               → [contratosElectronicos, procesosDeContratacion]


## 8. Próximos pasos

Con la tabla base definida (`contratosElectronicos`), el plan de trabajo es:

1. **Notebook 01**: Exploración de `contratosElectronicos` — descargar muestra de Obra, perfilar datos reales, detectar calidad
2. **Notebook 02**: Exploración de `procesosDeContratacion` — features pre-contrato
3. **Notebook 03**: Exploración de `adiciones` — construir features de modificaciones
4. **Notebook 04**: Exploración de `proveedoresRegistrados` — perfil de proveedor
5. **Notebook 05**: Integración — JOIN de fuentes, esquema final, feature engineering